In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import warnings
import sys
import os

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

from functions.tools import configure_notebook_display, load_raw_datasets, save_dataset

configure_notebook_display()

After analysing and finding all the inconsistency and noise, the following prepoocessing is applied.

In [2]:
# Updated tracker from notebook 04
issue_tracker = pd.DataFrame({

    "column": [
        "parcel_id",
        "sowing_date",
        "date",
        "ndvi_value",
        "sensor_status",
        "parcel_id",
        "parcel_id",
    ],

    "issue_identified": [
        "Parcel inconsistency between metadata and readings",
        "Stored as string instead of datetime",
        "Multiple date formats within the same column",
        "NDVI values outside valid biological range [-1, 1]",
        "Dirty categorical labels and missing statuses",
        "Incomplete parcel coverage",
        "Temporal continuity gaps"
    ],

    "severity": [
        "High",
        "Medium",
        "High",
        "High",
        "Medium",
        "High",
        "Medium"
    ],

    "preprocessing_method_applied": [

        "SEPARATED rogue parcel records into separate dataset and excluded from trusted readings",
        "CONVERTED into standardized datetime format",
        "CONVERTED and normalized mixed date formats",
        "FLAGGED invalid vegetation readings as CORRUPTED but not removed",
        "CONVERTED casing, trimmed whitespace, and mapped missing values to UNKNOWN",
        "KEPT as the observed parcel were rogue and on merging would be removed anyway",
        "KEPT as the observed parcel were rogue and on merging would be removed anyway"
    ],

})

display(issue_tracker)

,column,issue_identified,severity,preprocessing_method_applied
0,parcel_id,Parcel inconsistency between metadata and readings,High,SEPARATED rogue parcel records into separate dataset and excluded from trusted readings
1,sowing_date,Stored as string instead of datetime,Medium,CONVERTED into standardized datetime format
2,date,Multiple date formats within the same column,High,CONVERTED and normalized mixed date formats
3,ndvi_value,"NDVI values outside valid biological range [-1, 1]",High,FLAGGED invalid vegetation readings as CORRUPTED but not removed
4,sensor_status,Dirty categorical labels and missing statuses,Medium,"CONVERTED casing, trimmed whitespace, and mapped missing values to UNKNOWN"
5,parcel_id,Incomplete parcel coverage,High,KEPT as the observed parcel were rogue and on merging would be removed anyway
6,parcel_id,Temporal continuity gaps,Medium,KEPT as the observed parcel were rogue and on merging would be removed anyway


In [3]:
# ============================================================
# LOADING DATASETS
# ============================================================

df_meta, df_readings = load_raw_datasets()

### Applying Data Preprocessing after finalising the major issues with the data in the previous notebooks

In [4]:
# ============================================================
# PRESERVING RAW SENSOR STATUS
# ============================================================

df_readings["sensor_status_raw"] = df_readings["sensor_status"]

In [5]:
# ============================================================
# NORMALIZING DATES
# ============================================================

df_readings["date"] = pd.to_datetime(
    df_readings["date"],
    format="mixed",
    dayfirst=True)

df_meta["sowing_date"] = pd.to_datetime(
    df_meta["sowing_date"])

In [6]:
# ============================================================
# STANDARDIZING SENSOR STATUS
# ============================================================

df_readings["sensor_status"] = (
    df_readings["sensor_status"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.lower())

status_mapping = {
    "ok": "OK",
    "error": "ERROR",
    "degraded": "DEGRADED",
    "unknown": "UNKNOWN"}

df_readings["sensor_status"] = (
    df_readings["sensor_status"]
    .map(status_mapping)
    .fillna("UNKNOWN"))

In [7]:
print("Standardized Sensor Status Distribution:\n")

print(df_readings["sensor_status"]
    .value_counts(dropna=False))

Standardized Sensor Status Distribution:

sensor_status
OK         3064
ERROR       246
UNKNOWN     137
Name: count, dtype: int64


In [8]:
# ============================================================
# CREATING QUALITY OF THE OBSERVATION
# ============================================================

conditions = [

    (
        (df_readings["ndvi_value"] >= -1) &
        (df_readings["ndvi_value"] <= 1) &
        (df_readings["sensor_status"] == "OK")
    ),

    (
        (df_readings["ndvi_value"] >= -1) &
        (df_readings["ndvi_value"] <= 1) &
        (df_readings["sensor_status"] == "ERROR")
    ),

    (
        (df_readings["ndvi_value"] >= -1) &
        (df_readings["ndvi_value"] <= 1) &
        (df_readings["sensor_status"] == "UNKNOWN")
    ),

    (
        (df_readings["ndvi_value"] < -1) |
        (df_readings["ndvi_value"] > 1)
    )
]

observation_quality = [
    "TRUSTED",
    "DEGRADED",
    "UNKNOWN",
    "CORRUPTED"
    ]

df_readings["observation_quality"] = np.select(
    conditions,
    observation_quality,
    default="UNKNOWN"
    )

Instead of treating all non-OK readings equally, the preprocessing separates degraded observations from fully corrupted ones. This keeps operational uncertainty visible without discarding potentially meaningful signals. The observation quality framework is intentionally scoped around NDVI validity and operational sensor state.

Rows are classified using:
- NDVI physical range
- sensor operational status

This framework therefore reflects observational reliability for analytical use, not a complete guarantee of environmental correctness across all variables though.

In [9]:
print("Trust Level Distribution:\n")

print(df_readings["observation_quality"].value_counts())

Trust Level Distribution:

observation_quality
TRUSTED      3064
DEGRADED      142
UNKNOWN       137
CORRUPTED     104
Name: count, dtype: int64


The majority of readings remain trustworthy after validation. The results show that degraded sensor states do not always produce invalid NDVI values, which supports keeping degradation separate from corruption.

In [10]:
# ============================================================
# OBSERVATION QUALITY DISTRIBUTION
# ============================================================

trust_distribution = (
    df_readings["observation_quality"]
    .value_counts()
    .reset_index())

trust_distribution.columns = [
    "observation_quality",
    "count"
    ]

trust_distribution["percentage"] = (
    trust_distribution["count"] /
    len(df_readings)) * 100

display(trust_distribution)

,observation_quality,count,percentage
0,TRUSTED,3064,88.888889
1,DEGRADED,142,4.119524
2,UNKNOWN,137,3.974471
3,CORRUPTED,104,3.017116


Sensor observations are categorized into:
- TRUSTED -> physically valid and stable
- DEGRADED -> operationally unstable but still potentially usable
- CORRUPTED -> invalid readings
- UNKNOWN -> missing operational certainty

This classification supports confidence for preprocessing.

In [11]:
# ============================================================
# IDENTIFY ROGUE READING RECORDS
# ============================================================

metadata_parcels = set(df_meta["parcel_id"])

rogue_readings = df_readings[
    ~df_readings["parcel_id"].isin(metadata_parcels)
    ].copy()

display(rogue_readings.head())

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,sensor_status_raw,observation_quality
23,PARCEL_099,2026-03-19,0.577,15.6,0.0,OK,OK,TRUSTED
75,PARCEL_098,2026-02-01,0.038,19.6,4.1,OK,ok,TRUSTED
129,PARCEL_099,2026-03-26,0.654,32.9,1.7,OK,OK,TRUSTED
262,PARCEL_098,2026-01-27,0.181,18.9,6.0,OK,OK,TRUSTED
323,PARCEL_098,2026-06-04,0.441,21.2,4.4,OK,OK,TRUSTED


In [12]:
# ============================================================
# REMOVING ROGUE RECORDS FROM PRIMARY ANALYSIS
# ============================================================

trusted_readings = df_readings[
    df_readings["parcel_id"].isin(metadata_parcels)
    ].copy()

In [13]:
# ============================================================
# FINAL PREPROCESSED DATASET PREVIEW
# ============================================================

display(trusted_readings.head())

print("Final Dataset Shape:")
print(trusted_readings.shape)

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,sensor_status_raw,observation_quality
0,PARCEL_018,2026-05-16,0.595,15.4,0.0,ERROR,error,DEGRADED
1,PARCEL_014,2026-01-27,0.457,27.6,0.0,OK,OK,TRUSTED
2,PARCEL_006,2026-05-17,0.497,25.8,0.0,OK,OK,TRUSTED
3,PARCEL_004,2026-02-10,0.810,25.0,0.0,OK,OK,TRUSTED
4,PARCEL_002,2026-01-17,0.269,33.2,0.0,OK,OK,TRUSTED


Final Dataset Shape:
(3407, 8)


In [14]:
# ============================================================
# SAVE PREPROCESSED DATASETS
# ============================================================

# Main processed readings dataset
save_dataset(
    trusted_readings,
    "../data/trusted_readings.csv")

# Rogue readings quarantine dataset
save_dataset(
    rogue_readings,
    "../data/rogue_readings.csv")

# Processed metadata dataset
save_dataset(
    df_meta,
    "../data/processed_metadata.csv")


Dataset saved at: ../data/trusted_readings.csv
Dataset saved at: ../data/rogue_readings.csv
Dataset saved at: ../data/processed_metadata.csv
